# 17 Void catalogs from void voxels

DeepVoid labels every voxel; a catalog needs separate voids. The T-web void class percolates (one region spans the box), so every method needs a rule for where one void ends and the next begins. None is free of choices; each choice here is an explicit parameter, and the point of this notebook is to see what each choice does, on the true labels and on the predictions alike.

| Method | Rule | Parameter |
|---|---|---|
| Cores | void voxels above a score cut, or the void mask shrunk by k cells; separate pieces become voids, grown back over the void voxels | score cut or k |
| Basins | each void voxel climbs the distance to the nearest non-void voxel to a local maximum (a void center); a basin less than Δ deeper than the pass to a bigger one is merged into it | Δ (h⁻¹ Mpc) |
| Soft basins | as basins, on the distance averaged over several void-score cuts, so walls the model is unsure of separate voids less | Δ, the cuts |
| Spheres | the distance is the largest empty sphere at each voxel; the largest spheres that barely overlap are voids, smaller ones overlapping one enough join it (the idea of VoidFinder, not its algorithm, which works on galaxies) | minimum radius, overlaps |

The basins' merge history (section 4): for each candidate void center, how deep it is (the radius of the largest empty sphere there) against how wide the neck to a deeper neighbor is. Centers far below the diagonal are distinct voids; centers near it are dents in a bigger void. Δ is a cut on that difference. (In topological data analysis this plot is a persistence diagram, with the two axes called birth and death.)

**Data.** The whole box, truth and prediction: `BOX = 'tng'` (205 h⁻¹ Mpc, 512³, quicker) or `'bolshoi'` (250 h⁻¹ Mpc, 640³). The methods are defined on the true T-web labels; the predictions then go through the identical settings to see whether the same voids come out. The model is set by `MODEL`.

**Threshold.** `LAMBDA_TH` sets the T-web eigenvalue threshold. At 0.65 (the training labels) the saved mask is used; any other value is computed from the dark-matter density at σ_eff = 1 h⁻¹ Mpc (the eigenvalues are saved once and reused), truth only. Outputs are tagged with box and threshold; section 8 compares every threshold run so far.

**Truth only.** With `TRUTH_ONLY = True` (the default) everything runs on the true labels on a CPU runtime: thresholded cores by erosion, distance basins and spheres. The void-score cut and the soft distance need model scores and are skipped.

**Runtime.** No predicted volumes were saved before (earlier notebooks kept only scores), so section 2 predicts the whole box at every spacing on a GPU once and saves the four class scores per voxel to `deepvoid_runs/predictions/` (float16, ~1 GB per spacing for TNG), for this and any later notebook; reruns skip it. Everything else is CPU but needs **High-RAM** (the distance transform of a padded 640³ box takes ~12 GB, about half that for 512³). `COARSEN = 2` halves the grid for a quick look.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/deepvoid
!git clone -q "https://github.com/skumagai3/deepvoid.git" /content/deepvoid
!pip -q install --force-reinstall --no-deps /content/deepvoid
!git -C /content/deepvoid log --oneline -1

## 1 Settings

All the choices are here.

In [ ]:
import gc
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

from deepvoid import Catalog
from deepvoid import voids as V
from deepvoid.labels import VOID
from deepvoid.resample import block_mean

RUNS = Path('/content/drive/MyDrive/deepvoid_runs')
MODEL = 'subhalo_C_mixed_smooth2ch_depth5_distill'   # a run under deepvoid_runs, or 'release' for the
                                                     # published model (notebook 11)
OUT = RUNS / 'voids'
OUT.mkdir(exist_ok=True)
PRED = RUNS / 'predictions'                          # whole-box class scores, kept for reuse
PRED.mkdir(exist_ok=True)
catalog = Catalog()

BOX = 'tng'                 # 'tng' (512^3, quicker) or 'bolshoi' (640^3)
SOURCES = {'tng': ('tng300-3/mask', 'tng300-3/subhalos/L{s}', 'tng300-3/dm'),
           'bolshoi': ('bolshoi/mask', 'bolshoi/halos/L{s}', 'bolshoi/dm')}
LAMBDA_TH = 0.65            # T-web eigenvalue threshold; 0.65 is the training labels' value.
                            # Others are computed from the dark-matter density (truth only).
TRUTH_ONLY = True           # True: catalogs from the true labels only (CPU, no predictions)
PREDICT = [] if TRUTH_ONLY else [3, 5, 7, 10]    # spacings predicted and saved
SPACINGS = [] if TRUTH_ONLY else [5, 10]         # spacings used for catalogs below
TAG = f'{BOX}_th{LAMBDA_TH}' if TRUTH_ONLY else f'{BOX}_th{LAMBDA_TH}_{MODEL}'   # names the outputs
COARSEN = 1                 # 2: half the cells per side, for a quick look
MARGIN = 96                 # padding (cells) for the periodic distance transform
R_CUT = 3.0                 # h^-1 Mpc: voids listed and matched have R_eff above this

ERODE = []                              # cores: mask shrunk by k cells, e.g. [1, 2, 3] (off: slow, and
                                        # at λ_th = 0.65 one region keeps >99% of the void volume)
SCORE_CUTS = []                         # cores: void score cuts, e.g. [0.8, 0.95] (predictions only; off)
PERSISTENCE = [0.5, 1.0, 2.0]           # basins and soft basins: Δ in h^-1 Mpc
MIN_RADIUS = 1.0                        # basins and soft basins: drop voids whose largest empty sphere is smaller
SOFT_CUTS = (0.2, 0.5, 0.8)             # soft basins: void score cuts averaged over
SPHERES = [dict(r_min=5.0), dict(r_min=10.0)]   # spheres; overlaps 10% / 50% as VoidFinder

USE_CACHE = True            # reuse distances, basins, spheres and matches saved by an earlier run
REFRESH = False             # True: recompute everything and overwrite the cache
CACHE_VERSION = 1           # bump after changing deepvoid.voids, so old results are not reused
CACHE = OUT / 'cache'
CACHE.mkdir(exist_ok=True)


TIMINGS = RUNS / 'voids' / 'timings.json'   # how long each slow step took on earlier runs (shared by 17 and 18)


def _kind(name):
    # the same step at another spacing or box counts as an estimate: 'L10_...' -> 'L*_...'
    import re
    return re.sub(r'(^|_)L\d+', r'\1L*', name)


def timed(name, fn):
    # Runs fn(), prints how long the same step took last time (an estimate), and records this run's time.
    t = json.loads(TIMINGS.read_text()) if TIMINGS.exists() else {}
    if name in t:
        print(f'{name}: last time {t[name] / 60:.1f} min', flush=True)
    elif _kind(name) in t:
        print(f'{name}: about {t[_kind(name)] / 60:.1f} min at another spacing last time', flush=True)
    else:
        print(f'{name}: no earlier timing', flush=True)
    t0 = time.time()
    res = fn()
    dt = time.time() - t0
    t = json.loads(TIMINGS.read_text()) if TIMINGS.exists() else {}
    t[name] = dt
    t[_kind(name)] = dt
    TIMINGS.write_text(json.dumps(t, indent=1))
    print(f'{name}: {dt / 60:.1f} min', flush=True)
    return res


def cached(name, fn):
    # Result of fn(), saved to the Drive cache the first time and read back afterwards. name must say
    # everything the result depends on (box, threshold, model, spacing, parameters).
    path = CACHE / f'v{CACHE_VERSION}_{BOX}_th{LAMBDA_TH}_c{COARSEN}_{name}.npz'.replace(' ', '')
    if USE_CACHE and not REFRESH and path.exists():
        with np.load(path) as f:
            out = {k: f[k] for k in f.files}
        out = {k: v.astype(np.int32) if v.dtype == np.uint16 else v for k, v in out.items()}
        return out['_'] if list(out) == ['_'] else out
    res = timed(f'{BOX}_th{LAMBDA_TH}_{name}'.replace(' ', ''), fn)
    d = res if isinstance(res, dict) else {'_': res}
    d = {k: v.astype(np.uint16) if (np.issubdtype(np.asarray(v).dtype, np.integer) and np.asarray(v).ndim == 3
                                    and 0 <= v.min() and v.max() < 65535) else v for k, v in d.items()}
    if USE_CACHE:
        np.savez(path, **d)
    return res


def pred_name(s):
    return f'{MODEL}_L{s}'


def basin_labels(which, p, distance=None, soft=False):
    # Basins at persistence p, for the truth ('truth') or a spacing s; soft=True uses the soft distance.
    void = truth_void if which == 'truth' else pred[which]['void']
    d = distance if distance is not None else dist[which]
    tag = 'truth' if which == 'truth' else pred_name(which)
    name = f"{tag}_{'soft' if soft else ''}basins_p{p}_r{MIN_RADIUS}" + (f'_cuts{SOFT_CUTS}' if soft else '')
    return cached(name, lambda: V.distance_voids(void, distance=d, persistence=p, min_radius=MIN_RADIUS))

**Time estimates.** Every slow step that is cached (distances, basins, spheres, matches) prints how long the same step took on the last run, or at another spacing if this one is new, and records its own time in `deepvoid_runs/voids/timings.json`. The first run has no estimates.

**Cache.** Distances, persistence diagrams, basins, soft distances, spheres and void matches are saved to `deepvoid_runs/voids/cache/` the first time and read back on later runs with the same box, threshold, model and parameters (the file name holds them all). A first run fills it (several GB for TNG, about twice that for Bolshoi); later runs skip the slow steps. Set `REFRESH = True` to recompute, and bump `CACHE_VERSION` after changing `deepvoid.voids`, since the cache cannot tell that the code changed.

## 2 Truth and predictions

Whole-box predictions of `MODEL` at every spacing in `PREDICT` are saved once to `deepvoid_runs/predictions/<box>_<model>_L<spacing>.npz` (array `scores`, shape (n, n, n, 4), float16, the softmax scores; class = argmax) and read back afterwards. The default model is the distilled one, whose void scores are the best calibrated (notebook 05 runs), which matters for the score cuts and the soft basins.

In [ ]:
def coarse_mask(void):
    return block_mean(void.astype(np.float32), COARSEN) >= 0.5 if COARSEN > 1 else void


if LAMBDA_TH == 0.65:
    truth_vol = catalog.load(SOURCES[BOX][0])
    box = truth_vol.box_size
    truth_void = truth_vol.data == VOID
    del truth_vol
else:
    if not TRUTH_ONLY:
        raise ValueError('the model was trained on labels at 0.65; use TRUTH_ONLY for other thresholds')
    from deepvoid.labels import smoothing_sigma, tweb_eigenvalues
    eig_path = OUT / f'{BOX}_tweb_eigenvalues.npy'         # kept: each threshold reuses them
    dm = catalog.load(SOURCES[BOX][2])
    box = dm.box_size
    if eig_path.exists():
        eig = np.load(eig_path)
    else:
        t0 = time.time()
        eig = tweb_eigenvalues(dm.data, smoothing_sigma(dm.box_size, dm.n, 1.0))
        np.save(eig_path, eig)
        print(f'T-web eigenvalues in {time.time() - t0:.0f} s, saved {eig_path.name}')
    del dm
    truth_void = (eig > LAMBDA_TH).sum(-1) == VOID
    del eig
    gc.collect()
truth_void = coarse_mask(truth_void)
n = truth_void.shape[0]
cell = box / n
print(f'{BOX}, λ_th = {LAMBDA_TH}: {n}^3 cells of {cell:.3f} h^-1 Mpc; void share {truth_void.mean():.3f}')

model = None
for s in PREDICT:
    path = PRED / f'{BOX}_{MODEL}_L{s}.npz'
    if path.exists():
        continue
    if model is None:
        from deepvoid.pretrained import Pretrained
        model = (Pretrained.from_folder(RUNS.parent / 'deepvoid_model_release') if MODEL == 'release'
                 else Pretrained.from_run(RUNS / MODEL, catalog=catalog))
    vol = catalog.load(SOURCES[BOX][1].format(s=s))
    t0 = time.time()
    probs = model.predict(vol.data, vol.box_size, float(s), out_dtype=np.float16)
    np.savez(path, scores=probs, box_size=vol.box_size, spacing=float(s), model=MODEL)
    print(f'λ = {s}: predicted in {time.time() - t0:.0f} s, saved {path}', flush=True)
    del probs, vol
    gc.collect()

pred = {}
for s in SPACINGS:
    probs = np.load(PRED / f'{BOX}_{MODEL}_L{s}.npz')['scores']
    score = probs[..., VOID].astype(np.float32)
    void = probs.argmax(-1) == VOID
    del probs
    if COARSEN > 1:
        score = block_mean(score, COARSEN)
        void = coarse_mask(void)
    pred[s] = {'score': score, 'void': void}
    print(f'λ = {s}: void share {void.mean():.3f}, agreement with truth {np.mean(void == truth_void):.3f}')

## 3 Does the void class percolate?

Number of separate void regions (periodic) and the share of void voxels in the largest. If the largest holds nearly everything, connected components alone give one giant void.

In [ ]:
perc = {'truth': V.percolation(truth_void)}
for s in SPACINGS:
    perc[f'λ = {s}'] = V.percolation(pred[s]['void'])
print(pd.DataFrame(perc).T.to_string())

## 4 Distances and persistence diagrams

Distance from every void voxel to the nearest non-void voxel (h⁻¹ Mpc). Each local maximum of that distance is a candidate void center. Left panels: each candidate's depth (the distance at its center) against the width of the neck where it meets a deeper neighbor (the distance at the narrowest point between them). Right: the depth minus the neck width for every candidate. Δ in the basins and soft basins keeps candidates whose depth exceeds their neck width by more than Δ.

In [ ]:
t0 = time.time()
dist = {'truth': cached('truth_distance', lambda: V.distance_to_structure(truth_void, cell=cell, margin=MARGIN))}
for s in SPACINGS:
    dist[s] = cached(f'{pred_name(s)}_distance',
                     lambda: V.distance_to_structure(pred[s]['void'], cell=cell, margin=MARGIN))
print(f'distances in {time.time() - t0:.0f} s; largest:', {k: round(float(d.max()), 1) for k, d in dist.items()})

diagrams = {'truth': cached('truth_persistence', lambda: V.basin_persistence(truth_void, dist['truth']))}
for s in SPACINGS:
    diagrams[s] = cached(f'{pred_name(s)}_persistence', lambda: V.basin_persistence(pred[s]['void'], dist[s]))

fig, axes = plt.subplots(1, len(diagrams) + 1, figsize=(4.8 * (len(diagrams) + 1), 4.4))
cut_styles = [':', '--', '-.', (0, (1, 3))]
for ax, (k, h) in zip(axes, diagrams.items()):
    hb = ax.hexbin(h[:, 0], h[:, 1], gridsize=60, bins='log', mincnt=1, cmap='viridis')
    fig.colorbar(hb, ax=ax, label='candidate centers per hexagon')
    lim = max(h[:, 0].max(), 1)
    ax.plot([0, lim], [0, lim], 'k-', lw=0.8, label='depth = neck width')
    for p, ls in zip(PERSISTENCE, cut_styles):
        ax.plot([p, lim], [0, lim - p], color='r', ls=ls, lw=0.9, label=f'Δ = {p}: below is a separate void')
    ax.set_title(f'{BOX}, λ_th = {LAMBDA_TH}: ' + ('truth' if k == 'truth' else f'prediction, λ = {k}'))
    ax.set_xlabel('center depth: largest empty sphere (h⁻¹ Mpc)'); ax.set_ylabel('neck width to a deeper void (h⁻¹ Mpc)')
    ax.legend(fontsize=7, loc='upper left')
ax = axes[-1]
bins = np.logspace(-2, 1.3, 60)
for k, h in diagrams.items():
    ax.hist(h[:, 0] - h[:, 1], bins=bins, histtype='step', label='truth' if k == 'truth' else f'prediction, λ = {k}')
for p, ls in zip(PERSISTENCE, cut_styles):
    ax.axvline(p, color='r', ls=ls, lw=0.9, label=f'Δ = {p}')
ax.axvline(cell, color='0.5', ls='-', lw=0.8, label=f'one grid cell ({cell:.2f})')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('depth minus neck width (h⁻¹ Mpc)'); ax.set_ylabel('number of candidate centers')
ax.legend(fontsize=7)
ax.set_title('how distinct each candidate is')
fig.tight_layout()
fig.savefig(OUT / f'{TAG}_persistence.png', dpi=150, bbox_inches='tight')
np.savez(OUT / f'{TAG}_persistence.npz', **{str(k): h for k, h in diagrams.items()})
plt.show()

## 5 Catalogs from every method

Each method runs on the truth and on each prediction with the same settings. For each catalog: the number of voids with R_eff > `R_CUT`, their median R_eff, and, for the predictions, how they match the truth catalog made with the same settings:

- **completeness**: share of true voids (R_eff > `R_CUT`) whose best-overlapping predicted void shares more than half their union (Jaccard > 0.5);
- **purity**: the same from the predicted side;
- **offset**: median distance between matched centers, h⁻¹ Mpc.

For spheres, voids match when the predicted void's largest sphere is centered within half the true one's radius.

In [ ]:
def periodic_sep(a, b):
    d = a - b
    d -= box * np.round(d / box)
    return np.sqrt((d ** 2).sum(-1))


matches = {}   # (method, catalog) -> per-void sizes and whether each matched, for completeness by size
FATES = ('matched', 'merged', 'split or shrunk', 'missed')


def fates(lab_t, lab_p, tp):
    # What happened to each true void: 0 matched (Jaccard > 0.5); 1 merged (half or more of it lies in one
    # predicted void that is larger); 2 split or shrunk (half or more is predicted void, but not inside one
    # larger predicted void); 3 missed (less than half is predicted void).
    size_t = np.bincount(lab_t.ravel())[1:]
    size_p = np.bincount(lab_p.ravel(), minlength=int(lab_p.max()) + 1)
    in_void = np.bincount(lab_t.ravel(), weights=(lab_p > 0).ravel(), minlength=len(size_t) + 1)[1:]
    contain = tp['shared'] / np.maximum(size_t, 1)
    out = np.full(len(size_t), 2, np.int8)
    out[in_void < 0.5 * size_t] = 3
    out[(contain >= 0.5) & (size_p[tp['best']] > size_t)] = 1
    out[tp['jaccard'] > 0.5] = 0
    return out


def merge_aware(lab_t, lab_p, tp, fate):
    # Match each predicted void to the union of the true voids assigned to it (those matched to it or merged
    # into it). A predicted void matches when the Jaccard index of it and that union is above 0.5; every true
    # void in the union then counts as found. Returns per true void and per predicted void whether matched,
    # and per predicted void how many true voids its union holds.
    size_t = np.bincount(lab_t.ravel())[1:]
    n_p = int(lab_p.max())
    size_p = np.bincount(lab_p.ravel(), minlength=n_p + 1)[1:]
    assigned = (fate <= 1) & (tp['best'] > 0)
    j = tp['best'][assigned] - 1
    inter = np.bincount(j, weights=tp['shared'][assigned], minlength=n_p)
    union_t = np.bincount(j, weights=size_t[assigned], minlength=n_p)
    group = np.bincount(j, minlength=n_p)
    with np.errstate(invalid='ignore', divide='ignore'):
        jac = np.where(group > 0, inter / (size_p + union_t - inter), 0.0)
    ok_p = jac > 0.5
    ok_t = np.zeros(len(size_t), bool)
    ok_t[assigned] = ok_p[j]
    return ok_t, ok_p, group



def matched(key, lab_t, lab_p):
    # match_voids both ways (true -> predicted, predicted -> true), cached per method and spacing
    slug = f'{MODEL}_match_{key[0]}_{key[1]}'.replace(',', '').replace('=', '').replace('λ', 'L').replace('Δ', 'D')
    mm = cached(slug + f'_r{MIN_RADIUS}' + (f'_cuts{SOFT_CUTS}' if key[0].startswith('soft') else ''),
                lambda: {f'{a}_{k}': v for a, m in (('tp', V.match_voids(lab_t, lab_p)),
                                                    ('pt', V.match_voids(lab_p, lab_t))) for k, v in m.items()})
    return ({k: mm[f'tp_{k}'] for k in ('best', 'shared', 'jaccard')},
            {k: mm[f'pt_{k}'] for k in ('best', 'shared', 'jaccard')})


def compare(lab_t, lab_p, key):
    cat_t, cat_p = V.void_catalog(lab_t, cell), V.void_catalog(lab_p, cell)
    big_t, big_p = cat_t['r_eff'] > R_CUT, cat_p['r_eff'] > R_CUT
    tp, pt = matched(key, lab_t, lab_p)
    ok_t = big_t & (tp['jaccard'] > 0.5)
    ok_p = big_p & (pt['jaccard'] > 0.5)
    off = periodic_sep(cat_t['center'][ok_t], cat_p['center'][tp['best'][ok_t] - 1])
    fate = fates(lab_t, lab_p, tp)
    mt, mp, group = merge_aware(lab_t, lab_p, tp, fate)
    matches[key] = {'r_true': cat_t['r_eff'], 'ok_true': tp['jaccard'] > 0.5,
                    'r_pred': cat_p['r_eff'], 'ok_pred': pt['jaccard'] > 0.5, 'fate': fate,
                    'ok_true_merge': mt, 'ok_pred_merge': mp, 'group': group, 'best': tp['best']}
    row = {'voids': int(big_p.sum()), 'median R_eff': float(np.median(cat_p['r_eff'][big_p])) if big_p.any() else np.nan,
           'completeness': float(ok_t.sum() / max(big_t.sum(), 1)),
           'purity': float(ok_p.sum() / max(big_p.sum(), 1)),
           'offset': float(np.median(off)) if len(off) else np.nan}
    for i, f in enumerate(FATES[1:], start=1):
        row[f] = float(np.mean(fate[big_t] == i)) if big_t.any() else np.nan
    row['completeness, merge-aware'] = float(mt[big_t].mean()) if big_t.any() else np.nan
    row['purity, merge-aware'] = float(mp[big_p].mean()) if big_p.any() else np.nan
    ok_g = mp & big_p & (group > 0)
    row['true voids per matched predicted void'] = float(np.mean(group[ok_g])) if ok_g.any() else np.nan
    return cat_p, row


def summary(lab):
    cat = V.void_catalog(lab, cell)
    big = cat['r_eff'] > R_CUT
    return cat, {'voids': int(big.sum()), 'median R_eff': float(np.median(cat['r_eff'][big])) if big.any() else np.nan}


def methods():
    """(name, function of (void, score, distance)); score is None for the truth."""
    for k in ERODE:
        yield f'cores, eroded {k}', lambda v, sc, d, w, k=k: V.threshold_voids(v, erode=k)
    for c in SCORE_CUTS:
        yield f'cores, score ≥ {c}', lambda v, sc, d, w, c=c: None if sc is None else V.threshold_voids(v, score=sc, threshold=c)
    for p in PERSISTENCE:
        yield f'basins, Δ = {p}', lambda v, sc, d, w, p=p: basin_labels(w, p)


rows, radii = {}, {}
for name, run in methods():
    t0 = time.time()
    lab_t = run(truth_void, None, dist['truth'], 'truth')
    if lab_t is not None:
        cat_t, rows[(name, 'truth')] = summary(lab_t)
        radii[(name, 'truth')] = cat_t['r_eff']
    for s in SPACINGS:
        lab_p = run(pred[s]['void'], pred[s]['score'], dist[s], s)
        if lab_t is None:
            cat_p, rows[(name, f'λ = {s}')] = summary(lab_p)
        else:
            cat_p, rows[(name, f'λ = {s}')] = compare(lab_t, lab_p, (name, f'λ = {s}'))
        radii[(name, f'λ = {s}')] = cat_p['r_eff']
        del lab_p
    del lab_t
    gc.collect()
    print(f'{name}: {time.time() - t0:.0f} s', flush=True)

Soft basins: the soft distance exists only for predictions (the truth has no scores), so their catalogs are compared with the truth's basins at the same Δ. Its distances are averages over the cuts, so the same Δ is not quite the same depth.

In [ ]:
soft = {}
for s in SPACINGS:
    t0 = time.time()
    soft[s] = cached(f'{pred_name(s)}_softdistance_cuts{SOFT_CUTS}',
                     lambda: V.soft_distance(pred[s]['score'], thresholds=SOFT_CUTS, cell=cell, margin=MARGIN))
    print(f'λ = {s}: soft distance in {time.time() - t0:.0f} s', flush=True)
for p in PERSISTENCE if SPACINGS else []:
    lab_t = basin_labels('truth', p)
    for s in SPACINGS:
        lab_p = basin_labels(s, p, distance=soft[s], soft=True)
        key = (f'soft basins, Δ = {p}', f'λ = {s}')
        cat_p, rows[key] = compare(lab_t, lab_p, key)
        radii[(f'soft basins, Δ = {p}', f'λ = {s}')] = cat_p['r_eff']
        del lab_p
    del lab_t
    gc.collect()
del soft

In [ ]:
def sphere_match(sp_t, sp_p):
    """True voids whose largest sphere has a predicted largest sphere within half its radius."""
    def largest(sp):
        ids = np.unique(sp['void'])
        out_c, out_r = [], []
        for v in ids:
            sel = sp['void'] == v
            k = np.argmax(sp['radius'] * sel)
            out_c.append(sp['center'][k]); out_r.append(sp['radius'][k])
        return np.array(out_c).reshape(-1, 3), np.array(out_r)
    ct, rt = largest(sp_t)
    cp, rp = largest(sp_p)
    if not len(ct) or not len(cp):
        return 0.0, 0.0, np.nan, np.zeros(len(ct), bool), np.zeros(len(cp), bool)
    tree_p, tree_t = cKDTree(cp % box, boxsize=box), cKDTree(ct % box, boxsize=box)
    dt, _ = tree_p.query(ct % box)
    dp, _ = tree_t.query(cp % box)
    ok_t, ok_p = dt < rt / 2, dp < rp / 2
    off = float(np.median(dt[ok_t])) if ok_t.any() else np.nan
    return float(ok_t.mean()), float(ok_p.mean()), off, ok_t, ok_p


for kw in SPHERES:
    name = f"spheres, r_min = {kw['r_min']}"
    t0 = time.time()
    sp_tag = '_'.join(f'{k}{v}' for k, v in sorted(kw.items()))
    sp_t = cached(f'truth_spheres_{sp_tag}', lambda: V.sphere_voids(dist['truth'], truth_void, cell, **kw))
    vol_t = V.sphere_union_volume(sp_t)
    r_t = (3 * vol_t / (4 * np.pi)) ** (1 / 3)
    rows[(name, 'truth')] = {'voids': len(r_t), 'median R_eff': float(np.median(r_t)) if len(r_t) else np.nan}
    radii[(name, 'truth')] = r_t
    for s in SPACINGS:
        sp_p = cached(f'{pred_name(s)}_spheres_{sp_tag}', lambda: V.sphere_voids(dist[s], pred[s]['void'], cell, **kw))
        r_p = (3 * V.sphere_union_volume(sp_p) / (4 * np.pi)) ** (1 / 3)
        comp, pur, off, ok_t, ok_p = sphere_match(sp_t, sp_p)
        matches[(name, f'λ = {s}')] = {'r_true': r_t, 'ok_true': ok_t, 'r_pred': r_p, 'ok_pred': ok_p}
        rows[(name, f'λ = {s}')] = {'voids': len(r_p), 'median R_eff': float(np.median(r_p)) if len(r_p) else np.nan,
                                    'completeness': comp, 'purity': pur, 'offset': off}
        radii[(name, f'λ = {s}')] = r_p
    print(f'{name}: {time.time() - t0:.0f} s', flush=True)

## 6 Results

The size functions are cumulative, n(>R), on log axes running to the largest void found. A flat stretch means no voids in that range: below `r_min` for spheres, and between the ordinary voids and one box-spanning region when a method fails to split the void class (the last step is then that one region).

In [ ]:
table = pd.DataFrame(rows).T
table.index.names = ['method', 'catalog']
pd.set_option('display.width', 160)
print(table.round(2).to_string())
(OUT / f'{TAG}_results.json').write_text(json.dumps(
    {f'{m}|{c}': {k: (None if pd.isna(v) else v) for k, v in r.items()} for (m, c), r in rows.items()}, indent=1))
np.savez(OUT / f'{TAG}_radii.npz', **{f'{m}|{c}': r for (m, c), r in radii.items()})

In [ ]:
names = list(dict.fromkeys(m for m, _ in radii))
ncol = 4
nrow = int(np.ceil(len(names) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4.2 * ncol, 3.4 * nrow), sharex=True, sharey=True)
r_top = max(float(np.max(r)) for r in radii.values() if len(r)) * 1.05
bins = np.logspace(np.log10(R_CUT), np.log10(r_top), 80)   # up to the largest void, however big
styles = {'truth': ('k', '-')}
styles.update({f'λ = {s}': (c, '--') for s, c in zip(SPACINGS, ['#dd6b20', '#2b6cb0', '#38a169'])})
for ax, name in zip(axes.ravel(), names):
    for (m, c), r in radii.items():
        if m == name:
            col, ls = styles[c]
            ax.step(bins, V.size_function(r, box, bins), where='post', color=col, ls=ls, label=c)
    plain = name.replace('soft basins', 'basins')
    if name.startswith('soft basins') and (plain, 'truth') in radii:   # soft basins: truth from basins
        ax.step(bins, V.size_function(radii[(plain, 'truth')], box, bins), where='post',
                color='k', ls='-', label='truth (basins)')
    ax.set_xscale('log'); ax.set_yscale('log'); ax.set_title(name, fontsize=10)
    ticks = [t for t in (3, 5, 10, 20, 50, 100, 200) if t <= r_top]
    ax.set_xticks(ticks, [str(t) for t in ticks]); ax.minorticks_off()
for ax in axes.ravel()[len(names):]:
    ax.axis('off')
for ax in axes.ravel()[:len(names)]:
    ax.tick_params(labelbottom=True)
    ax.set_xlabel(r'$R_{\rm eff}$ ($h^{-1}$ Mpc)')
for ax in axes[:, 0]:
    ax.set_ylabel(r'$n(>R)$ ($h^3$ Mpc$^{-3}$)')
for ax in axes.ravel()[:len(names)]:
    if ax.get_lines():
        ax.legend(fontsize=7)
    else:
        ax.text(0.5, 0.5, f'no voids with R_eff > {R_CUT}', transform=ax.transAxes, ha='center')
fig.suptitle(f'Void size functions, {BOX}, λ_th = {LAMBDA_TH}' + ('' if TRUTH_ONLY else ': truth against predictions, same settings'))
fig.tight_layout()
fig.savefig(OUT / f'{TAG}_size_functions.png', dpi=150, bbox_inches='tight')
plt.show()

### Completeness and purity by void size

The table above pools voids of every size. Here they are split by size: for each method and spacing, the share of true voids matched (completeness, against the true void's R_eff) and the share of predicted voids matched (purity, against the predicted void's R_eff), with the same matching as above. Bins with fewer than `MIN_PER_BIN` voids are left out. The dotted vertical line marks 2λ. Void finders that work on tracers of mean spacing λ produce spurious (Poisson) voids at small radii, and void size function analyses cut their catalogs there; Ronconi et al. (2019, arXiv:1902.04585) cut at twice the mean inter-particle separation. Our voids come from T-web labels rather than from the tracers, so the line is a reference: to its right a tracer void finder's catalog would also be trusted.

In [ ]:
MIN_PER_BIN = 10
names_m = list(dict.fromkeys(m for m, _ in matches))
if not names_m:
    print('no matched catalogs (truth-only run)')
else:
    r_top = max(float(np.max(v['r_true'])) for v in matches.values() if len(v['r_true'])) * 1.05
    edges = np.logspace(np.log10(R_CUT), np.log10(r_top), 14)
    mid = np.sqrt(edges[1:] * edges[:-1])
    colors = {f'λ = {s}': c for s, c in zip(SPACINGS, ['#dd6b20', '#2b6cb0', '#38a169'])}

    def binned(r, ok):
        idx = np.digitize(r, edges) - 1
        out = np.full(len(mid), np.nan)
        for i in range(len(mid)):
            sel = idx == i
            if sel.sum() >= MIN_PER_BIN:
                out[i] = ok[sel].mean()
        return out

    ncol = 3
    nrow = int(np.ceil(len(names_m) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(5.2 * ncol, 3.8 * nrow), squeeze=False)
    for ax, name in zip(axes.ravel(), names_m):
        for s in SPACINGS:
            key = (name, f'λ = {s}')
            if key not in matches:
                continue
            m = matches[key]
            col = colors[f'λ = {s}']
            ax.plot(mid, binned(m['r_true'], m['ok_true']), '-o', ms=3, color=col, label=f'λ = {s}: completeness')
            ax.plot(mid, binned(m['r_pred'], m['ok_pred']), '--s', ms=3, color=col, label=f'λ = {s}: purity')
            ax.axvline(2 * s, color=col, ls=':', lw=1.5, label=f'λ = {s}: 2λ = {2 * s}')
        ax.set_xscale('log'); ax.set_ylim(0, 1.02)
        ticks = [t for t in (3, 5, 10, 20, 50, 100) if t <= r_top]
        ax.set_xticks(ticks, [str(t) for t in ticks]); ax.minorticks_off()
        ax.set_xlabel(r'void $R_{\rm eff}$ ($h^{-1}$ Mpc): true for completeness, predicted for purity')
        ax.set_ylabel('share of voids matched')
        ax.set_title(name, fontsize=10)
        ax.grid(alpha=0.3)
    for ax in axes.ravel()[len(names_m):]:
        ax.axis('off')
    fig.suptitle(f'{BOX}, λ_th = {LAMBDA_TH}, model {MODEL}: matching by void size '
                 f'(Jaccard > 0.5; spheres: centers within half the radius)')
    handles, labels = axes.ravel()[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='lower center', ncol=len(SPACINGS) * 3, fontsize=9,
               title='solid: completeness (true voids found); dashed: purity (predicted voids that are real); dotted: 2λ',
               bbox_to_anchor=(0.5, -0.04))
    fig.tight_layout()
    fig.savefig(OUT / f'{TAG}_completeness_by_size.png', dpi=150, bbox_inches='tight')
    plt.show()
    np.savez(OUT / f'{TAG}_matches.npz', **{f'{m}|{c}|{k}': v for (m, c), d in matches.items() for k, v in d.items()})

### Missed or merged?

The Jaccard rule counts a merge as a miss on both sides: if one predicted void covers five true voids, none of the six is matched. Here every true void above `R_CUT` gets one fate:

- **matched:** Jaccard > 0.5, as above;
- **merged:** not matched, and half or more of it lies inside one predicted void that is larger than it;
- **split or shrunk:** not matched, half or more of its volume is predicted void, but not inside one larger predicted void;
- **missed:** less than half of its volume is predicted void.

Shown for the basins at the middle Δ; the shares for every method are in the results table (columns merged, split or shrunk, missed; matched = completeness).

In [ ]:
P_MID = PERSISTENCE[len(PERSISTENCE) // 2]
keys = [(f'basins, Δ = {P_MID}', f'λ = {s}') for s in SPACINGS if (f'basins, Δ = {P_MID}', f'λ = {s}') in matches]
if not keys:
    print('no matched catalogs (truth-only run)')
else:
    fate_colors = {'matched': '#2b6cb0', 'merged': '#dd6b20', 'split or shrunk': '#38a169', 'missed': '#9b2c2c'}
    edges_f = np.logspace(np.log10(R_CUT), np.log10(max(float(matches[k]['r_true'].max()) for k in keys) * 1.05), 12)
    mid_f = np.sqrt(edges_f[1:] * edges_f[:-1])
    fig, axes = plt.subplots(1, len(keys), figsize=(6.5 * len(keys), 4.5), squeeze=False)
    for ax, key in zip(axes[0], keys):
        m = matches[key]
        idx = np.digitize(m['r_true'], edges_f) - 1
        s = float(key[1].split('= ')[1])
        for i, f in enumerate(FATES):
            share = np.full(len(mid_f), np.nan)
            for b in range(len(mid_f)):
                sel = idx == b
                if sel.sum() >= MIN_PER_BIN:
                    share[b] = np.mean(m['fate'][sel] == i)
            ax.plot(mid_f, share, '-o', ms=3, color=fate_colors[f], label=f)
        ax.axvline(2 * s, color='k', ls=':', lw=1.2, label='2λ')
        ax.set_xscale('log'); ax.set_ylim(0, 1.02); ax.set_xlim(edges_f[0], edges_f[-1])
        ticks = [t for t in (3, 5, 10, 20, 50) if t <= edges_f[-1]]
        ax.set_xticks(ticks, [str(t) for t in ticks]); ax.minorticks_off()
        ax.set_xlabel(r'true void $R_{\rm eff}$ ($h^{-1}$ Mpc)')
        ax.set_ylabel('share of true voids')
        ax.set_title(f'{key[0]}, {key[1]}')
        ax.grid(alpha=0.3)
    h, l = axes[0, 0].get_legend_handles_labels()
    fig.legend(h, l, loc='lower center', ncol=len(l), fontsize=10, frameon=False, bbox_to_anchor=(0.5, -0.06))
    fig.suptitle(f'{BOX}, λ_th = {LAMBDA_TH}, model {MODEL}: what happened to each true void, by size '
                 f'(bins with ≥ {MIN_PER_BIN} voids)')
    fig.tight_layout()
    fig.savefig(OUT / f'{TAG}_fates_by_size.png', dpi=150, bbox_inches='tight')
    plt.show()
    for key in keys:
        m = matches[key]
        s = float(key[1].split('= ')[1])
        for lo, sel in (('all', m['r_true'] > R_CUT), ('above 2λ', m['r_true'] > 2 * s)):
            counts = np.bincount(m['fate'][sel], minlength=4)
            print(f'{key[1]}, {lo} ({sel.sum()} voids): '
                  + ', '.join(f'{f} {c / max(sel.sum(), 1):.2f}' for f, c in zip(FATES, counts)))

### Merge-aware matching

Most unmatched true voids at large λ are merged, not missed. Merge-aware matching asks whether each predicted void is a fair outline of a group of true voids: the union of the true voids matched to it or merged into it (the fates above). A predicted void matches when the Jaccard index of it and that union is above 0.5, and every true void in the union then counts as found. Strict matching (section 5) is the special case of groups of one.

This measures how well the predictions recover void complexes: what a catalog at that spacing can claim if it reports them as such. The table above has the columns `completeness, merge-aware`, `purity, merge-aware` and `true voids per matched predicted void` (mean group size).

In [ ]:
if not keys:
    print('no matched catalogs (truth-only run)')
else:
    edges_m = np.logspace(np.log10(R_CUT), np.log10(max(float(matches[k]['r_true'].max()) for k in keys) * 1.05), 12)
    mid_m = np.sqrt(edges_m[1:] * edges_m[:-1])

    def by_size(r, ok):
        idx = np.digitize(r, edges_m) - 1
        out = np.full(len(mid_m), np.nan)
        for b in range(len(mid_m)):
            sel = idx == b
            if sel.sum() >= MIN_PER_BIN:
                out[b] = ok[sel].mean()
        return out

    fig, axes = plt.subplots(1, len(keys), figsize=(6.5 * len(keys), 4.5), squeeze=False)
    for ax, key in zip(axes[0], keys):
        m = matches[key]
        s = float(key[1].split('= ')[1])
        ax.plot(mid_m, by_size(m['r_true'], m['ok_true']), '-o', ms=3, color='#2b6cb0', label='completeness, strict')
        ax.plot(mid_m, by_size(m['r_true'], m['ok_true_merge']), '-o', ms=3, color='#dd6b20', label='completeness, merge-aware')
        ax.plot(mid_m, by_size(m['r_pred'], m['ok_pred']), '--s', ms=3, color='#2b6cb0', label='purity, strict')
        ax.plot(mid_m, by_size(m['r_pred'], m['ok_pred_merge']), '--s', ms=3, color='#dd6b20', label='purity, merge-aware')
        ax.axvline(2 * s, color='k', ls=':', lw=1.2, label='2λ')
        ax.set_xscale('log'); ax.set_ylim(0, 1.02); ax.set_xlim(edges_m[0], edges_m[-1])
        ticks = [t for t in (3, 5, 10, 20, 50) if t <= edges_m[-1]]
        ax.set_xticks(ticks, [str(t) for t in ticks]); ax.minorticks_off()
        ax.set_xlabel(r'void $R_{\rm eff}$ ($h^{-1}$ Mpc): true for completeness, predicted for purity')
        ax.set_ylabel('share of voids matched')
        ax.set_title(f'{key[0]}, {key[1]}')
        ax.grid(alpha=0.3)
    h, l = axes[0, 0].get_legend_handles_labels()
    fig.legend(h, l, loc='upper center', ncol=len(l), fontsize=10, frameon=False, bbox_to_anchor=(0.5, 0.0))
    fig.suptitle(f'{BOX}, λ_th = {LAMBDA_TH}, model {MODEL}: strict and merge-aware matching, by size '
                 f'(Jaccard > 0.5; bins with ≥ {MIN_PER_BIN} voids)')
    fig.tight_layout()
    fig.savefig(OUT / f'{TAG}_merge_aware_by_size.png', dpi=150, bbox_inches='tight')
    plt.show()
    cols = ['completeness', 'completeness, merge-aware', 'purity', 'purity, merge-aware',
            'true voids per matched predicted void']
    print(pd.DataFrame({f'{a} | {c}': {k: rows[(a, c)][k] for k in cols}
                        for (a, c) in rows if c != 'truth' and 'completeness, merge-aware' in rows[(a, c)]}).T.round(2).to_string())

## 7 What the voids look like

One slice through the box, x = middle, basins at the middle Δ.

- **Truth only:** the true basins at each Δ, one random color per void.
- **With predictions:** one row per spacing, four panels. The dark-matter density and the true voids are the same in every row (they don't depend on λ); what changes with λ is the subhalo input and the prediction.
  - Left: dark-matter density in the slice, the outlines of the true and predicted voids, and the cells holding a subhalo (the CIC-gridded input) in a slab of thickness λ around the slice: what the model saw.
  - Middle two: the true and predicted voids, filled. A predicted void matched to a true void (Jaccard > 0.5, the rule in section 5) takes that void's color, so the same color in both panels is the same void. Unmatched voids are white with a black edge.
  - Right: a 70 h⁻¹ Mpc zoom of the left panel (dashed box) on the predicted void that swallowed the most true voids.

Matching is done in 3D. A slice can cut a matched pair at different heights, so a pair can look different here and still match.

In [ ]:
from matplotlib.colors import ListedColormap
from scipy import ndimage
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

SLICE = n // 2
P_MID = PERSISTENCE[len(PERSISTENCE) // 2]
STRUCT = np.array([0.30, 0.30, 0.30])     # wall, filament, halo
UNMATCHED = np.array([1.0, 1.0, 1.0])
EDGE_T, EDGE_P = np.array([0.0, 0.75, 1.0]), np.array([1.0, 0.35, 0.0])   # outline colors
TRACER = np.array([0.1, 0.85, 0.1])      # cells holding a subhalo
ZOOM = 70.0                              # h^-1 Mpc, side of the zoomed panel
DENSITY_ALPHA = 0.7                      # opacity of the density background (1 = full)


def basins(which):
    return basin_labels(which, P_MID)


def edges(sl):
    # pixels on the boundary of a void (next to structure or to another void), periodic
    e = np.zeros(sl.shape, bool)
    for ax in (0, 1):
        for sh in (1, -1):
            e |= sl != np.roll(sl, sh, axis=ax)
    return e & (sl > 0)


def fill(sl, colors, matched_edge):
    img = colors[sl]
    img[edges(sl) & matched_edge[sl]] = 0.0    # black edge on unmatched voids
    return img.transpose(1, 0, 2)


rng = np.random.default_rng(0)
lab_t = basins('truth')
if not SPACINGS:
    panels = [(P_MID, lab_t)] + [(p, basin_labels('truth', p)) for p in PERSISTENCE if p != P_MID]
    panels.sort(key=lambda x: x[0])
    fig, axes = plt.subplots(1, len(panels), figsize=(6 * len(panels), 6), squeeze=False)
    for ax, (p, lab) in zip(axes[0], panels):
        sl = lab[SLICE]
        colors = rng.random((sl.max() + 1, 3)) * 0.8 + 0.2
        colors[0] = STRUCT
        ax.imshow(colors[sl].transpose(1, 0, 2), origin='lower', extent=[0, box, 0, box])
        ax.set_title(f'true basins, Δ = {p} h⁻¹ Mpc')
        ax.set_xlabel('y (h⁻¹ Mpc)'); ax.set_ylabel('z (h⁻¹ Mpc)')
    handles = [Patch(color=STRUCT, label='wall, filament or halo (not void)'),
               Patch(facecolor='none', edgecolor='none', label='every other color is one void (random colors)')]
else:
    dm = catalog.load(SOURCES[BOX][2])
    k = max(int(COARSEN), 1)
    rho = dm.data[SLICE * k:(SLICE + 1) * k].mean(0)
    if k > 1:
        rho = block_mean(rho, k)
    rho3 = block_mean(dm.data, k) if k > 1 else dm.data
    log_rho3d = np.log10(np.maximum(rho3 / rho3.mean(), 1e-2)).astype(np.float32)   # kept for section 7b
    del dm, rho3
    basin_lab = {'truth': lab_t}                                                       # kept for section 7b
    log_rho = np.log10(np.maximum(rho / rho.mean(), 1e-2))
    n_t = int(lab_t.max())
    col_t = rng.random((n_t + 1, 3)) * 0.75 + 0.25
    col_t[0] = STRUCT
    fig, axes = plt.subplots(len(SPACINGS), 4, figsize=(24, 6 * len(SPACINGS)), squeeze=False)
    for row, s in zip(axes, SPACINGS):
        lab_p = basins(s)
        tp, pt = matched((f'basins, Δ = {P_MID}', f'λ = {s}'), lab_t, lab_p)
        fate = fates(lab_t, lab_p, tp)
        ok_t = np.r_[True, tp['jaccard'] > 0.5]           # index 0 = structure
        ok_p = np.r_[True, pt['jaccard'] > 0.5]
        ct = col_t.copy(); ct[~ok_t] = UNMATCHED
        cp = np.tile(UNMATCHED, (int(lab_p.max()) + 1, 1)); cp[0] = STRUCT
        j = np.flatnonzero(ok_p[1:]) + 1
        cp[j] = col_t[pt['best'][j - 1]]
        st, sp = lab_t[SLICE], lab_p[SLICE]

        vol = catalog.load(SOURCES[BOX][1].format(s=s))
        h = max(int(round(s / 2 / (box / vol.data.shape[0]))), 0)
        c0 = SLICE * vol.data.shape[0] // n
        slab = np.take(vol.data, np.arange(c0 - h, c0 + h + 1), axis=0, mode='wrap')
        occ = (slab > slab.min()).any(0)
        if occ.shape[0] != n:
            occ = block_mean(occ.astype(np.float32), occ.shape[0] // n) > 0
        del vol, slab

        over = np.zeros((n, n, 4))              # zoom: one pixel per subhalo cell
        over[occ] = (*TRACER, 1.0)
        over[edges(st)] = (*EDGE_T, 1.0)
        over[edges(sp)] = (*EDGE_P, 1.0)
        over_full = over.copy()                 # full slice: subhalo cells grown to 3x3 so they show
        big = ndimage.binary_dilation(occ, structure=np.ones((3, 3), bool))
        over_full[big & ~(edges(st) | edges(sp))] = (*TRACER, 1.0)
        # zoom: the predicted void in this slice that swallowed the most true voids
        merged_into = np.bincount(tp['best'][fate == 1], minlength=int(lab_p.max()) + 1)
        in_slice = np.unique(sp[sp > 0])
        j = in_slice[np.argmax(merged_into[in_slice])] if len(in_slice) else 0
        yy, zz = np.nonzero(sp == j) if j else (np.array([n // 2]), np.array([n // 2]))
        half = min(ZOOM, box / 2) / 2
        cy = np.clip(np.median(yy) * cell, half, box - half)
        cz = np.clip(np.median(zz) * cell, half, box - half)
        for ax, zoom in ((row[0], False), (row[3], True)):
            im = ax.imshow(log_rho.T, origin='lower', extent=[0, box, 0, box], cmap='gray_r', vmin=-1, vmax=1.5,
                           alpha=DENSITY_ALPHA)
            ax.imshow((over if zoom else over_full).transpose(1, 0, 2), origin='lower', extent=[0, box, 0, box],
                      interpolation='nearest')
            if zoom:
                ax.set_xlim(cy - half, cy + half); ax.set_ylim(cz - half, cz + half)
                ax.set_title(f'λ = {s}: zoom on a predicted void that merged {merged_into[j]} true voids\n(counted in the whole box; fewer may cross this slice)'
                             if merged_into[j] else f'λ = {s}: zoom (no merged voids in this slice)', fontsize=10)
                fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02,
                             label=r'dark-matter density, $\log_{10}(\rho/\bar\rho)$ in the slice')
            else:
                ax.add_patch(plt.Rectangle((cy - half, cz - half), 2 * half, 2 * half, fill=False, ec='k', lw=1.5, ls='--'))
                ax.set_title(f'λ = {s}: density, void outlines, subhalos within ±{s / 2:g} h⁻¹ Mpc')
        row[1].imshow(fill(st, ct, ~ok_t), origin='lower', extent=[0, box, 0, box], interpolation='nearest')
        row[1].set_title(f'true basins (matched to λ = {s} in color)')
        row[2].imshow(fill(sp, cp, ~ok_p), origin='lower', extent=[0, box, 0, box], interpolation='nearest')
        row[2].set_title(f'predicted basins, λ = {s}')
        for a in row:
            a.set_xlabel('y (h⁻¹ Mpc)'); a.set_ylabel('z (h⁻¹ Mpc)')
        basin_lab[s] = lab_p
    handles = [Line2D([], [], color=EDGE_T, lw=2, label='true void outline (left, zoom)'),
               Line2D([], [], color=EDGE_P, lw=2, label='predicted void outline (left, zoom; drawn over the true one where they coincide)'),
               Patch(color=TRACER, label='cell holding a subhalo (CIC grid), slab of thickness λ (left: enlarged to 3×3 cells; zoom: true size)'),
               Patch(facecolor='none', edgecolor='k', ls='--', label='zoomed region (left)'),
               Patch(color=STRUCT, label='wall, filament or halo (middle panels)'),
               Patch(facecolor='none', edgecolor='none', label='same color = matched true/predicted pair (middle panels)'),
               Patch(facecolor=UNMATCHED, edgecolor='k', label='unmatched void (middle panels)')]
fig.suptitle(f'{BOX}, λ_th = {LAMBDA_TH}' + ('' if TRUTH_ONLY else f', model {MODEL}')
             + f': slice x = {SLICE * cell:.0f} h⁻¹ Mpc' + ('' if not SPACINGS else f', basins at Δ = {P_MID} h⁻¹ Mpc'))
fig.tight_layout(rect=(0, 0, 1, 0.97))
fig.legend(handles=handles, loc='upper center', ncol=2 if SPACINGS else 1, fontsize=10, frameon=False,
           bbox_to_anchor=(0.5, 0.0))
fig.savefig(OUT / f'{TAG}_slices.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved in', OUT)

## 7b Individual voids

Four views of single voids (basins at the middle Δ, as in section 7), needing predictions:

1. **Gallery:** true voids of a few sizes, each cut through its center along three planes, with the true void's outline and, for each spacing, the outline of the predicted void that overlaps it most.
2. **One merge in 3D:** the predicted void at the largest spacing that swallowed the most true voids, as a translucent hull, with the true voids inside it as colored surfaces. Interactive (rotate, zoom); saved as html.
3. **Size against size:** predicted R_eff against true R_eff for strictly matched pairs, and for merge-aware matches the predicted void against the union of its true voids.
4. **Shapes:** axis ratios from each void's inertia tensor (a ≥ b ≥ c), for the truth and each spacing.

In [ ]:
SPACING_COLORS = dict(zip(SPACINGS, ['#dd6b20', '#6b46c1', '#38a169']))
TRUE_COLOR = '#00a3e0'
basin_key = lambda s: (f'basins, Δ = {P_MID}', f'λ = {s}')
have_pred = bool(SPACINGS) and 'basin_lab' in globals() and all(basin_key(s) in matches for s in SPACINGS)
if not have_pred:
    print('needs predictions (TRUTH_ONLY = False) and section 7 run first')
else:
    cat_tb = V.void_catalog(basin_lab['truth'], cell)
    cat_pb = {s: V.void_catalog(basin_lab[s], cell) for s in SPACINGS}

In [ ]:
# 1 Gallery
GALLERY_R = [5.0, 10.0, 15.0, 20.0]       # h^-1 Mpc: one true void near each size
if have_pred:
    lab_t3 = basin_lab['truth']
    planes = [('x', 'y', 2), ('x', 'z', 1), ('y', 'z', 0)]    # (horizontal, vertical, axis cut)

    def window(arr, c, h, axis):
        idx = [(int(round(ci)) + np.arange(-h, h + 1)) % n for ci in c]
        idx[axis] = np.array([int(round(c[axis])) % n])
        return np.squeeze(arr[np.ix_(*idx)], axis=axis)      # 2D, (first remaining axis, second)

    rr = cat_tb['r_eff']
    picks = []
    for r0 in GALLERY_R:
        i = int(np.argmin(np.abs(rr - r0)))
        if i not in picks:
            picks.append(i)
    fig, axes = plt.subplots(len(picks), 3, figsize=(13, 4.2 * len(picks)), squeeze=False)
    for row, i in zip(axes, picks):
        c = cat_tb['center'][i] / cell
        h = min(int(np.ceil(max(1.8 * rr[i], 10.0) / cell)), n // 2 - 1)   # never wider than the box
        ext = [-h * cell, h * cell, -h * cell, h * cell]
        fate_txt = ', '.join(f'λ = {s}: {FATES[matches[basin_key(s)]["fate"][i]]}' for s in SPACINGS)
        for ax, (hu, hv, axis) in zip(row, planes):
            ax.imshow(window(log_rho3d, c, h, axis).T, origin='lower', extent=ext, cmap='gray_r',
                      vmin=-1, vmax=1.5, alpha=DENSITY_ALPHA)
            ax.contour(window(lab_t3 == i + 1, c, h, axis).T.astype(float), levels=[0.5], extent=ext,
                       colors=[TRUE_COLOR], linewidths=2.2)
            for s in SPACINGS:
                j = matches[basin_key(s)]['best'][i]
                if j > 0:
                    ax.contour(window(basin_lab[s] == j, c, h, axis).T.astype(float), levels=[0.5], extent=ext,
                               colors=[SPACING_COLORS[s]], linewidths=1.5)
            ax.set_xlabel(f'{hu} − {hu}$_c$ ($h^{{-1}}$ Mpc)'); ax.set_ylabel(f'{hv} − {hv}$_c$ ($h^{{-1}}$ Mpc)')
            ax.set_title(f'cut through the center, {hu}{hv} plane', fontsize=9)
            ax.set_aspect('equal')
        row[0].annotate(f'true void, R_eff = {rr[i]:.1f} h⁻¹ Mpc\n{fate_txt}', xy=(0, 1.13), xycoords='axes fraction',
                        fontsize=10, va='bottom')
    handles = [Line2D([], [], color=TRUE_COLOR, lw=2.2, label='true void')]
    handles += [Line2D([], [], color=SPACING_COLORS[s], lw=1.5, label=f'predicted void overlapping it most, λ = {s}')
                for s in SPACINGS]
    handles += [Patch(color='0.5', label=r'dark-matter density, $\log_{10}(\rho/\bar\rho)$ from −1 (white) to 1.5 (black)')]
    fig.suptitle(f'{BOX}, λ_th = {LAMBDA_TH}, model {MODEL}: true voids of several sizes and their predicted counterparts\n'
                 f'(basins, Δ = {P_MID} h⁻¹ Mpc; fates as in section 6)', y=1.0)
    fig.tight_layout()
    fig.legend(handles=handles, loc='upper center', ncol=2, fontsize=10, frameon=False, bbox_to_anchor=(0.5, 0.0))
    fig.savefig(OUT / f'{TAG}_gallery.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# 2 One merge in 3D (interactive)
if have_pred:
    import plotly.graph_objects as go
    from skimage.measure import marching_cubes

    STEP_3D = 2          # surface resolution in cells (1 = finest; larger files)
    s3 = SPACINGS[-1]
    m = matches[basin_key(s3)]
    members_of = m['best'] * ((m['fate'] <= 1) & (m['best'] > 0))       # predicted void each true void belongs to
    count = np.bincount(members_of[m['fate'] == 1], minlength=int(basin_lab[s3].max()) + 1)
    count[0] = 0
    j = int(np.argmax(count))
    members = np.flatnonzero(members_of == j) + 1                         # true void ids in this group
    cj = cat_pb[s3]['center'][j - 1] / cell
    shift = [n // 2 - int(round(ci)) for ci in cj]                         # move the void to the middle (periodic)
    P = np.roll(basin_lab[s3] == j, shift, axis=(0, 1, 2))
    T = np.roll(basin_lab['truth'], shift, axis=(0, 1, 2))
    lo = np.maximum(np.argwhere(P).min(0) - 3, 0)
    hi = np.minimum(np.argwhere(P).max(0) + 4, n)
    sl = tuple(slice(a, b) for a, b in zip(lo, hi))
    P, T = P[sl], T[sl]
    off = (lo - n // 2) * cell                                             # coordinates relative to the center

    def mesh(mask, color, opacity, name):
        v, f, _, _ = marching_cubes(np.pad(mask, 1).astype(np.float32), 0.5, step_size=STEP_3D)
        v = np.round((v - 1) * cell + off, 2).astype(np.float64)
        return go.Mesh3d(x=v[:, 0], y=v[:, 1], z=v[:, 2], i=f[:, 0], j=f[:, 1], k=f[:, 2],
                         color=color, opacity=opacity, name=name, showlegend=True, flatshading=True)

    rng3 = np.random.default_rng(1)
    traces = [mesh(P, 'lightgray', 0.18, f'predicted void (λ = {s3})')]
    for t in members:
        mask = T == t
        if mask.sum() > 8:
            r_t = cat_tb['r_eff'][t - 1]
            col = 'rgb({},{},{})'.format(*(rng3.integers(40, 230, 3)))
            traces.append(mesh(mask, col, 0.9, f'true void {t}, R_eff = {r_t:.1f}'))
    jac = 'yes' if m['ok_pred_merge'][j - 1] else 'no'
    fig3 = go.Figure(traces)
    half3 = float(np.max(np.abs(np.r_[off, off + (hi - lo) * cell]))) + 2.0      # one cube that holds everything
    ax3 = dict(range=[-half3, half3], dtick=20, autorange=False, showbackground=False, showspikes=False,
               gridcolor='rgba(0,0,0,0.15)', zeroline=False)
    fig3.update_layout(
        title=dict(text=(f'{BOX}, λ_th = {LAMBDA_TH}, λ = {s3}: predicted void {j} '
                         f'(R_eff = {cat_pb[s3]["r_eff"][j - 1]:.1f} h⁻¹ Mpc, gray hull)<br>'
                         f'and the {len(members)} true voids assigned to it ({count[j]} merged, colored); '
                         f'merge-aware match: {jac}'), font=dict(size=14)),
        scene=dict(xaxis=dict(title='x − x_c (h⁻¹ Mpc)', **ax3), yaxis=dict(title='y − y_c (h⁻¹ Mpc)', **ax3),
                   zaxis=dict(title='z − z_c (h⁻¹ Mpc)', **ax3), aspectmode='cube',
                   camera=dict(projection=dict(type='orthographic'))),
        uirevision='keep', legend=dict(title='surfaces (click to hide)', x=1.0, y=0.95),
        margin=dict(l=0, r=0, t=70, b=0), width=1050, height=800)
    fig3.write_html(OUT / f'{TAG}_merge_3d.html', include_plotlyjs='cdn')
    fig3.show()
    del P, T

In [ ]:
# 3 Size against size
if have_pred:
    fig, axes = plt.subplots(1, len(SPACINGS), figsize=(5.5 * len(SPACINGS), 6.5), squeeze=False)
    lim = [R_CUT, max(float(cat_tb['r_eff'].max()), max(float(cat_pb[s]['r_eff'].max()) for s in SPACINGS)) * 1.1]
    for ax, s in zip(axes[0], SPACINGS):
        m = matches[basin_key(s)]
        ok = m['ok_true'] & (m['r_true'] > R_CUT)
        x1, y1 = m['r_true'][ok], m['r_pred'][m['best'][ok] - 1]
        grp = (m['group'] > 1) & m['ok_pred_merge'] & (m['r_pred'] > R_CUT)
        assigned = (m['fate'] <= 1) & (m['best'] > 0)
        vol = np.bincount(m['best'][assigned] - 1, weights=m['r_true'][assigned] ** 3, minlength=len(m['r_pred']))
        x2, y2 = np.cbrt(vol[grp]), m['r_pred'][grp]
        ax.scatter(x1, y1, s=8, alpha=0.5, color=SPACING_COLORS[s], lw=0,
                   label=f'strictly matched pair ({len(x1)}); median ratio {np.median(y1 / x1):.2f}' if len(x1) else 'strictly matched pair (0)')
        ax.scatter(x2, y2, s=22, alpha=0.8, facecolor='none', edgecolor='k', lw=0.8,
                   label=f'merge-aware match, union of ≥ 2 true voids ({len(x2)})')
        ax.plot(lim, lim, 'k--', lw=1, label='predicted = true')
        ax.axvline(2 * s, color='k', ls=':', lw=1.2, label='2λ (true size)')
        ax.set_xscale('log'); ax.set_yscale('log'); ax.set_xlim(lim); ax.set_ylim(lim)
        ticks = [t for t in (3, 5, 10, 20, 50) if lim[0] <= t <= lim[1]]
        ax.set_xticks(ticks, [str(t) for t in ticks]); ax.set_yticks(ticks, [str(t) for t in ticks]); ax.minorticks_off()
        ax.set_xlabel(r'true $R_{\rm eff}$ ($h^{-1}$ Mpc; union of the group for merges)')
        ax.set_ylabel(r'predicted $R_{\rm eff}$ ($h^{-1}$ Mpc)')
        ax.set_title(f'basins, Δ = {P_MID}, λ = {s}')
        ax.legend(fontsize=8, loc='upper center', bbox_to_anchor=(0.5, -0.15), frameon=False)
        ax.set_aspect('equal'); ax.grid(alpha=0.3)
    fig.suptitle(f'{BOX}, λ_th = {LAMBDA_TH}, model {MODEL}: predicted against true void size')
    fig.tight_layout()
    fig.savefig(OUT / f'{TAG}_size_vs_size.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# 4 Shapes: axis ratios from the inertia tensor (periodic, about each void's catalog center)
def axis_ratios(lab, centers):
    k = int(lab.max())
    c = centers / cell
    mom = np.zeros((k + 1, 6))
    cnt = np.bincount(lab.ravel(), minlength=k + 1).astype(float)
    from tqdm.auto import tqdm
    yy, zz = np.meshgrid(np.arange(n), np.arange(n), indexing='ij')
    for x in tqdm(range(n), desc='inertia tensors', unit='plane', leave=False):   # one plane at a time
        l = lab[x].ravel()
        sel = l > 0
        l = l[sel]
        d = [x - c[l - 1, 0], yy.ravel()[sel] - c[l - 1, 1], zz.ravel()[sel] - c[l - 1, 2]]
        d = [di - n * np.round(di / n) for di in d]
        for q, (a, b) in enumerate([(0, 0), (1, 1), (2, 2), (0, 1), (0, 2), (1, 2)]):
            mom[:, q] += np.bincount(l, weights=d[a] * d[b], minlength=k + 1)
    mom = mom[1:] / np.maximum(cnt[1:, None], 1)
    I = np.stack([mom[:, [0, 3, 4]], mom[:, [3, 1, 5]], mom[:, [4, 5, 2]]], axis=1)
    ev = np.sort(np.clip(np.linalg.eigvalsh(I), 0, None), axis=1)[:, ::-1]
    a, b, cc = np.sqrt(ev).T
    with np.errstate(invalid='ignore', divide='ignore'):
        return b / a, cc / a


if have_pred:
    shape = {'truth': axis_ratios(basin_lab['truth'], cat_tb['center'])}
    big = {'truth': cat_tb['r_eff'] > R_CUT}
    for s in SPACINGS:
        shape[s] = axis_ratios(basin_lab[s], cat_pb[s]['center'])
        big[s] = cat_pb[s]['r_eff'] > R_CUT
    bins = np.linspace(0, 1, 26)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for ax, q, lab in zip(axes, (0, 1), ('b/a (middle / longest axis)', 'c/a (shortest / longest axis)')):
        for key, col, name in [('truth', TRUE_COLOR, 'truth')] + [(s, SPACING_COLORS[s], f'prediction, λ = {s}') for s in SPACINGS]:
            v = shape[key][q][big[key]]
            v = v[np.isfinite(v)]
            ax.hist(v, bins=bins, density=True, histtype='step', lw=2, color=col,
                    label=f'{name}: median {np.median(v):.2f} ({len(v)} voids)')
        ax.set_xlabel(f'axis ratio {lab}')
        ax.set_ylabel('probability density')
        ax.set_title(f'{lab.split(" ")[0]} distribution')
        ax.legend(fontsize=8, loc='upper center', bbox_to_anchor=(0.5, -0.17), frameon=False)
        ax.grid(alpha=0.3)
    fig.suptitle(f'{BOX}, λ_th = {LAMBDA_TH}, model {MODEL}: void shapes from the inertia tensor '
                 f'(basins, Δ = {P_MID} h⁻¹ Mpc, R_eff > {R_CUT:g} h⁻¹ Mpc; 1 = round)')
    fig.tight_layout()
    fig.savefig(OUT / f'{TAG}_shapes.png', dpi=150, bbox_inches='tight')
    plt.show()

## 8 Across thresholds

Every threshold run so far for this box with this version of the notebook (rerun with another `LAMBDA_TH` to add one): the depth-minus-neck distribution of the truth, and the truth size functions for the basins. If a lower threshold gives separate voids, the depth-minus-neck distribution should show a gap.

The left end of the depth-minus-neck histogram is jagged by construction: distances on a grid take only values √(i² + j² + k²) cells, so differences below about one cell come in fixed steps (0.17, 0.13, 0.11 … h⁻¹ Mpc for TNG's 0.40 cells).

In [ ]:
runs = [p for p in OUT.glob(f'{BOX}_th*_persistence.npz')
        if p.stem.count('_') == 2]                     # truth-only runs (no model in the name)
runs = sorted(runs, key=lambda p: float(p.stem.split('_th')[1].split('_')[0]))
ths = [float(p.stem.split('_th')[1].split('_')[0]) for p in runs]
if not runs:
    print('no saved runs yet')
else:
    fig, axes = plt.subplots(1, 1 + len(PERSISTENCE), figsize=(4.4 * (1 + len(PERSISTENCE)), 3.8))
    bins = np.logspace(-2, 1.3, 60)
    for th, path in zip(ths, runs):
        h = np.load(path)['truth']
        axes[0].hist(h[:, 0] - h[:, 1], bins=bins, histtype='step', label=f'λ_th = {th}')
        rpath = OUT / f'{BOX}_th{th}_radii.npz'
        if rpath.exists():
            radii_th = np.load(rpath)
            for ax, p in zip(axes[1:], PERSISTENCE):
                key = f'basins, Δ = {p}|truth'
                if key in radii_th:
                    rb = np.logspace(np.log10(R_CUT), np.log10(60), 80)
                    ax.step(rb, V.size_function(radii_th[key], box, rb), where='post', label=f'λ_th = {th}')
    axes[0].set_xscale('log'); axes[0].set_yscale('log'); axes[0].legend(fontsize=8)
    axes[0].set_xlabel('depth minus neck width (h⁻¹ Mpc)'); axes[0].set_ylabel('number of candidate centers')
    axes[0].set_title(f'{BOX} truth: how distinct each candidate is')
    for ax, p in zip(axes[1:], PERSISTENCE):
        ax.set_xscale('log'); ax.set_yscale('log'); ax.set_title(f'truth, basins, Δ = {p}')
        ax.set_xticks([3, 5, 10, 20, 50], ['3', '5', '10', '20', '50']); ax.minorticks_off()
        ax.set_xlabel(r'$R_{\rm eff}$ ($h^{-1}$ Mpc)'); ax.legend(fontsize=8)
    axes[1].set_ylabel(r'$n(>R)$ ($h^3$ Mpc$^{-3}$)')
    fig.tight_layout()
    fig.savefig(OUT / f'{BOX}_thresholds.png', dpi=150, bbox_inches='tight')
    plt.show()

## Shut down

Deletes this runtime once every cell above has finished, so it stops using compute units. The results are already on Drive; anything kept only under `/content` goes with the runtime. With **Run all**, Colab stops at the first cell that fails, so after an error the runtime stays up and you can look at it. Set `SHUT_DOWN = False` to keep the runtime.

In [ ]:
SHUT_DOWN = True   # False keeps the runtime after the run
if SHUT_DOWN:
    import time
    from google.colab import drive, runtime
    drive.flush_and_unmount()   # finish writing to Drive first
    time.sleep(5)
    runtime.unassign()          # disconnects and deletes this runtime